# 03 — Gộp Batch & EDA Trước Xử Lý

**Mục tiêu:**
1. Gộp tất cả file batch thành 4 file tổng (1 NSMO + 3 weather).
2. Kiểm tra gaps (timestamp bị thiếu) để đánh giá chất lượng thu thập.
3. EDA trước xử lý — phân tích phân phối, outlier và tương quan sơ bộ để định hướng bước làm sạch.

**Input:**
- `data/raw/NSMO/*.csv`
- `data/raw/Weather_Hanoi_Raw/*.csv`
- `data/raw/Weather_DaNang_Raw/*.csv`
- `data/raw/Weather_TPHCM_Raw/*.csv`

**Output:**
- `data/processed/nsmo_raw_merged.csv`
- `data/processed/hanoi_raw_merged.csv`
- `data/processed/danang_raw_merged.csv`
- `data/processed/hcm_raw_merged.csv`
- Biểu đồ EDA $\rightarrow$ `reports/eda_before/`

In [1]:
import os
import glob
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Repository Root
REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

# Constants
RAW_NSMO_PATH = Path('data/raw/NSMO')
RAW_WEATHER_PATHS = {
    'hanoi': Path('data/raw/Weather_Hanoi_Raw'),
    'danang': Path('data/raw/Weather_DaNang_Raw'),
    'hcm': Path('data/raw/Weather_TPHCM_Raw'),
}
PROCESSED_DIR = Path('data/processed')
REPORTS_DIR = Path('reports/eda_before')

LOAD_COLS = ['Load_North', 'Load_Central', 'Load_South', 'Load_National']
PRICE_COLS = ['Price_North', 'Price_Central', 'Price_South', 'Price_National']

# Create directories
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Plotting style
plt.rcParams.update({'figure.dpi': 120})
sns.set_theme(style='whitegrid')

print(f'[OK] Working directory: {os.getcwd()}')
print(f'[OK] Directories initialized: {PROCESSED_DIR}, {REPORTS_DIR}')

[OK] Working directory: c:\Users\ASUS\DS108-Project
[OK] Directories initialized: data\processed, reports\eda_before


## 1. Hàm tiện ích tái sử dụng

In [2]:
def merge_csv_batches(folder_path: Path, output_filename: str) -> pd.DataFrame:
    """Gộp tất cả file CSV trong thư mục, sắp xếp theo thời gian và loại bỏ trùng lặp.

    Args:
        folder_path: Đường dẫn đến thư mục chứa các file batch CSV.
        output_filename: Tên file kết quả lưu trong PROCESSED_DIR.

    Returns:
        DataFrame đã được gộp và làm sạch cơ bản.
    """
    all_files = sorted(folder_path.glob('*.csv'))
    assert len(all_files) > 0, f'No CSV files found in {folder_path}'

    df_list = [pd.read_csv(f, encoding='utf-8-sig') for f in all_files]
    merged_df = pd.concat(df_list, ignore_index=True)

    # Chuẩn hóa timestamp và loại bỏ trùng lặp
    ts_col = 'Timestamp' if 'Timestamp' in merged_df.columns else 'datetime'
    merged_df[ts_col] = pd.to_datetime(merged_df[ts_col])
    merged_df = (
        merged_df
        .drop_duplicates(subset=[ts_col])
        .sort_values(ts_col)
        .reset_index(drop=True)
    )

    # Lưu kết quả
    out_path = PROCESSED_DIR / output_filename
    merged_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    
    print(f'  → Saved: {output_filename} ({merged_df.shape[0]:,} rows x {merged_df.shape[1]} cols)')
    return merged_df


def check_time_gaps(df: pd.DataFrame, ts_col: str, freq: str) -> pd.DataFrame:
    """Kiểm tra các khoảng trống (missing timestamps) trong dữ liệu.

    Args:
        df: DataFrame chứa dữ liệu thời gian.
        ts_col: Tên cột mốc thời gian.
        freq: Tần suất dự kiến (ví dụ: '30min', 'h').

    Returns:
        DataFrame liệt kê các mốc thời gian bị thiếu.
    """
    expected_range = pd.date_range(
        start=df[ts_col].min(),
        end=df[ts_col].max(),
        freq=freq
    )
    missing = expected_range.difference(df[ts_col])
    return pd.DataFrame({'missing_timestamp': missing})


def plot_distribution(df: pd.DataFrame, cols: list, label: str, filename: str) -> None:
    """Vẽ histogram và boxplot để phân tích phân phối và outlier.

    Args:
        df: DataFrame chứa dữ liệu.
        cols: Danh sách các cột numeric cần vẽ.
        label: Nhãn định danh cho dữ liệu (ví dụ: 'NSMO', 'Hanoi').
        filename: Tên file ảnh lưu trong REPORTS_DIR.
    """
    n_cols = len(cols)
    fig, axes = plt.subplots(n_cols, 2, figsize=(12, 4 * n_cols))
    
    if n_cols == 1:
        axes = np.expand_dims(axes, axis=0)
        
    for i, col in enumerate(cols):
        # Histogram
        sns.histplot(df[col], kde=True, ax=axes[i, 0], color='steelblue')
        axes[i, 0].set_title(f'{label} - {col} Distribution')
        
        # Boxplot
        sns.boxplot(x=df[col], ax=axes[i, 1], color='lightcoral')
        axes[i, 1].set_title(f'{label} - {col} Outliers')
        
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / filename, bbox_inches='tight', dpi=120)
    plt.close()
    print(f'  → Plot saved: {filename}')

## 2. Gộp dữ liệu batch

In [3]:
print('[INFO] Starting data merging process...')

# Merge NSMO
df_nsmo = merge_csv_batches(RAW_NSMO_PATH, 'nsmo_raw_merged.csv')

# Merge Weather for 3 cities
df_weather_results = {}
for city, path in RAW_WEATHER_PATHS.items():
    filename = f'{city}_raw_merged.csv'
    df_weather_results[city] = merge_csv_batches(path, filename)

print('[OK] All batches merged successfully.')

[INFO] Starting data merging process...
  → Saved: nsmo_raw_merged.csv (52,608 rows x 9 cols)
  → Saved: hanoi_raw_merged.csv (26,304 rows x 24 cols)
  → Saved: danang_raw_merged.csv (26,304 rows x 24 cols)
  → Saved: hcm_raw_merged.csv (26,304 rows x 24 cols)
[OK] All batches merged successfully.


## 3. Kiểm tra gaps (Timestamp Missing)

In [4]:
print('[INFO] Checking for time gaps...')

# Check NSMO gaps (30min freq)
gaps_nsmo = check_time_gaps(df_nsmo, 'Timestamp', '30min')
print(f'  → NSMO: {len(gaps_nsmo)} missing intervals')

# Check Weather gaps (1h freq)
for city, df in df_weather_results.items():
    gaps = check_time_gaps(df, 'datetime', 'h')
    print(f'  → {city.capitalize()}: {len(gaps)} missing intervals')

print('[OK] Gap check completed.')

[INFO] Checking for time gaps...
  → NSMO: 0 missing intervals
  → Hanoi: 0 missing intervals
  → Danang: 0 missing intervals
  → Hcm: 0 missing intervals
[OK] Gap check completed.


## 4. EDA trước xử lý

In [5]:
print('[INFO] Generating initial EDA plots...')

# EDA for NSMO: Load and Price
plot_distribution(df_nsmo, LOAD_COLS, 'NSMO', 'eda_nsmo_load.png')
plot_distribution(df_nsmo, PRICE_COLS, 'NSMO', 'eda_nsmo_price.png')

# EDA for Weather: Temperature and Humidity (example columns)
weather_cols = ['temp', 'humidity']
for city, df in df_weather_results.items():
    plot_distribution(df, weather_cols, city.capitalize(), f'eda_weather_{city}.png')

print('[OK] EDA plots generated in reports/eda_before/')

[INFO] Generating initial EDA plots...
  → Plot saved: eda_nsmo_load.png
  → Plot saved: eda_nsmo_price.png
  → Plot saved: eda_weather_hanoi.png
  → Plot saved: eda_weather_danang.png
  → Plot saved: eda_weather_hcm.png
[OK] EDA plots generated in reports/eda_before/


## 5. Tổng kết và Validation

In [6]:
def validate_merging():
    """Kiểm tra nhanh tính toàn vẹn của dữ liệu sau khi gộp."""
    # Check if output files exist
    files_to_check = ['nsmo_raw_merged.csv', 'hanoi_raw_merged.csv', 
                      'danang_raw_merged.csv', 'hcm_raw_merged.csv']
    
    for f in files_to_check:
        assert (PROCESSED_DIR / f).exists(), f'Error: {f} was not created'
    
    # Check for duplicate timestamps in NSMO
    assert df_nsmo['Timestamp'].duplicated().sum() == 0, 'Duplicate timestamps found in NSMO'
    
    print('[OK] Final validation passed: All files exist and no duplicates.')

validate_merging()

[OK] Final validation passed: All files exist and no duplicates.
